# Domain gate — turbine / solar / invalid

A detector cannot say "that's a cat". Shown an out-of-domain photo it emits confident boxes
anyway, and reporting those as inspection findings is worse than useless. This small
classifier sits in front of both detectors and refuses anything it cannot place.

MobileNetV3-Small at 224px: about 2 MB after int8, a few minutes to train, and accurate
enough for a three-way domain decision. Using anything larger here would be spending the
visitor's bandwidth on the easiest problem in the pipeline.

### Data

| Class | Source | Roughly |
|---|---|---|
| `turbine` | Your existing 7,520 images — finally the right job for the `Healthy_*` sets | 3,000 |
| `solar` | Same RGB solar sets you train the solar detector on | 2,000 |
| `invalid` | Anything else: people, animals, buildings, vehicles, landscapes, screenshots, blurs | 3,000+ |

The `invalid` class matters most and is the one people skimp on. Make it **diverse** — a
gate trained against only cat photos will happily accept a photo of a roof. Include a few
hard negatives that look superficially similar: metal structures, glass facades, blue
rectangles, aerial farmland.

In [ ]:
!pip install -q torch torchvision onnx onnxruntime
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("cuda:", torch.cuda.is_available())

In [ ]:
from pathlib import Path

# Expected layout - one folder per class:
#   /kaggle/input/gate-data/train/{turbine,solar,invalid}/*.jpg
#   /kaggle/input/gate-data/val/{turbine,solar,invalid}/*.jpg
DATA = Path("/kaggle/input/gate-data")
CLASSES = ["turbine", "solar", "invalid"]   # order MUST match models/manifest.json

for split in ("train", "val"):
    for name in CLASSES:
        folder = DATA / split / name
        print(f"{split}/{name:<10}", len(list(folder.glob('*'))) if folder.exists() else "MISSING")

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

SIZE = 224
# These must match web/js/preprocess.js centerCrop(). If they drift, the browser feeds the
# model differently from how it was trained and accuracy quietly collapses.
NORMALIZE = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
    transforms.ToTensor(), NORMALIZE,
])
val_tf = transforms.Compose([
    transforms.Resize(SIZE), transforms.CenterCrop(SIZE),
    transforms.ToTensor(), NORMALIZE,
])

train_ds = datasets.ImageFolder(DATA / "train", train_tf)
val_ds = datasets.ImageFolder(DATA / "val", val_tf)
assert train_ds.classes == CLASSES, f"class order is {train_ds.classes}, must be {CLASSES}"

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=64, num_workers=2, pin_memory=True)
print(len(train_ds), "train /", len(val_ds), "val")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(CLASSES))
model = model.to(device)

# Class-weighted loss: the invalid class is usually the largest and would otherwise dominate.
counts = torch.bincount(torch.tensor(train_ds.targets), minlength=len(CLASSES)).float()
weights = (counts.sum() / (len(CLASSES) * counts)).to(device)
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

EPOCHS = 12
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)
best = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    for images, labels in train_dl:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        criterion(model(images), labels).backward()
        optimizer.step()
    scheduler.step()

    model.eval()
    confusion = torch.zeros(len(CLASSES), len(CLASSES), dtype=torch.long)
    with torch.no_grad():
        for images, labels in val_dl:
            predicted = model(images.to(device)).argmax(1).cpu()
            for true, pred in zip(labels, predicted):
                confusion[true, pred] += 1

    accuracy = confusion.diag().sum().item() / confusion.sum().item()
    print(f"epoch {epoch:>2}  val acc {accuracy:.4f}")
    if accuracy > best:
        best = accuracy
        torch.save(model.state_dict(), "/kaggle/working/gate_best.pt")

print(f"\nbest {best:.4f}")
print("\nconfusion (rows = true, cols = predicted):")
print(f"{'':<10}" + "".join(f"{c:>10}" for c in CLASSES))
for i, name in enumerate(CLASSES):
    print(f"{name:<10}" + "".join(f"{v:>10}" for v in confusion[i].tolist()))

### Read the confusion matrix before shipping

The cell that matters is **real defect image predicted as `invalid`** — that is the gate
refusing legitimate work, which users experience as the site being broken. Prefer that over
the opposite (`invalid` accepted as turbine/solar), but keep it low.

If `invalid` recall is high but real images get refused, your `invalid` set is too close to
the real classes, or too small. Add more diverse negatives rather than lowering the
threshold — `minConfidence` in the manifest is a blunt instrument.

In [ ]:
import torch, onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

model.load_state_dict(torch.load("/kaggle/working/gate_best.pt"))
model.eval().cpu()

torch.onnx.export(
    model, torch.randn(1, 3, SIZE, SIZE), "/kaggle/working/gate_fp32.onnx",
    input_names=["images"], output_names=["logits"],
    opset_version=12,          # what onnxruntime-web's WebGPU backend prefers
    dynamic_axes=None,         # static shapes are markedly faster under WASM
)

quantize_dynamic("/kaggle/working/gate_fp32.onnx", "/kaggle/working/gate.onnx",
                 weight_type=QuantType.QUInt8)

import os
print("fp32", os.path.getsize("/kaggle/working/gate_fp32.onnx") / 1e6, "MB")
print("int8", os.path.getsize("/kaggle/working/gate.onnx") / 1e6, "MB")

In [ ]:
# Sanity check the exported graph reproduces the torch model, before it reaches a browser.
import numpy as np, onnxruntime as ort

session = ort.InferenceSession("/kaggle/working/gate.onnx")
sample = torch.randn(1, 3, SIZE, SIZE)
with torch.no_grad():
    reference = model(sample).numpy()
exported = session.run(None, {"images": sample.numpy()})[0]

print("max abs difference:", np.abs(reference - exported).max())
print("same argmax:", reference.argmax() == exported.argmax())
print("\nCopy gate.onnx to web/models/ and confirm manifest labels are", CLASSES)